# Gizli Dirichlet Dağılımı (Latent Dirichlet Allocation) (LDA)

🎯 Bu challenge'ın amacı, **LDA** algoritması (NLP'de Denetimsiz Öğrenme) ile e-posta külliyatı içinde konular bulmaktır.

✉️ İşte 1000'den fazla ***etiketlenmemiş e-posta*** içeren bir koleksiyon. Bunlardan ***konuları çıkarmaya*** çalışalım!

In [1]:
import pandas as pd

url = 'https://d32aokrjazspmn.cloudfront.net/materials/lda_data'

data = pd.read_csv(url, sep=",", header=None)
data.columns = ['text']
data.head()

,text
0,From: gld@cunixb.cc.columbia.edu (Gary L Dare)...
1,From: atterlep@vela.acs.oakland.edu (Cardinal ...
2,From: miner@kuhub.cc.ukans.edu\nSubject: Re: A...
3,From: atterlep@vela.acs.oakland.edu (Cardinal ...
4,From: vzhivov@superior.carleton.ca (Vladimir Z...


In [2]:
data.shape

(1199, 1)

## (1) Preprocessing 

❓ **Question (Cleaning**) ❓ You're used to it by now... Clean up! Store the cleaned text in a new column "clean_text" of the DataFrame.

In [8]:
import string
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower().strip()
    text = "".join(char for char in text if not char.isdigit())
    text = text.translate(str.maketrans("", "", string.punctuation))
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return " ".join(tokens)

data["clean_text"] = data["text"].apply(clean_text)

data[["text", "clean_text"]].head()

,text,clean_text
0,From: gld@cunixb.cc.columbia.edu (Gary L Dare)...,gldcunixbcccolumbiaedu gary l dare subject sta...
1,From: atterlep@vela.acs.oakland.edu (Cardinal ...,atterlepvelaacsoaklandedu cardinal ximenez sub...
2,From: miner@kuhub.cc.ukans.edu\nSubject: Re: A...,minerkuhubccukansedu subject ancient book orga...
3,From: atterlep@vela.acs.oakland.edu (Cardinal ...,atterlepvelaacsoaklandedu cardinal ximenez sub...
4,From: vzhivov@superior.carleton.ca (Vladimir Z...,vzhivovsuperiorcarletonca vladimir zhivov subj...


## (2) Latent Dirichlet Allocation model

❓ **Soru (Eğitim)** ❓ Potansiyel konuları çıkarmak için bir LDA modeli eğitin

In [9]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

vectorizer = CountVectorizer(max_df=0.95, min_df=2, stop_words="english")
X = vectorizer.fit_transform(data["clean_text"])

lda = LatentDirichletAllocation(n_components=5, random_state=42)
lda.fit(X)

LatentDirichletAllocation(n_components=5, random_state=42)

##  (3) Potansiyel konuları görselleştirin

🎁 Potansiyel konularla ilişkili kelimeleri yazdırmak için bir  fonksiyon kodladık.

In [10]:
def print_topics(model, vectorizer):
    for idx, topic in enumerate(model.components_):
        print("Topic %d:" % (idx))
        print([(vectorizer.get_feature_names_out()[i], topic[i])
                        for i in topic.argsort()[:-10 - 1:-1]])

❓ **Soru** ❓ LDA tarafından çıkarılan konuları yazdırın.

In [11]:
print_topics(lda, vectorizer)

Topic 0:
[('team', 234.2047029108845), ('organization', 220.8474010592973), ('hockey', 209.324232645442), ('game', 205.08460431044423), ('university', 188.88088234271808), ('writes', 174.0087505728731), ('think', 172.14953937487866), ('time', 164.90542305820182), ('player', 156.45333316507165), ('year', 154.92705864212024)]
Topic 1:
[('team', 572.2153313107532), ('player', 343.59895029024045), ('game', 306.52449059277154), ('hockey', 294.9371559584109), ('nhl', 279.6530929255072), ('season', 274.41430179416267), ('organization', 236.36968340750497), ('play', 233.51046479685658), ('year', 230.6631446386747), ('league', 199.00701398385087)]
Topic 2:
[('god', 1327.481828261288), ('christian', 568.3548799040288), ('people', 527.4200315731927), ('jesus', 467.45527930177593), ('know', 401.4063039463439), ('organization', 382.74636774732835), ('think', 381.6596274347381), ('say', 378.46581643997825), ('believe', 374.3682161409628), ('dont', 323.6370944708205)]
Topic 3:
[('game', 415.461910408

## (4) Yeni bir metnin belge-konu karışımını tahmin edin

❓ **Soru (Tahmin)** ❓

LDA modeliniz fit edildiğine göre, onu yeni bir metnin konularını tahmin etmek için kullanabilirsiniz.

1. Örneği vektörleştirin
2. Vektörleştirilmiş örnek üzerinde LDA'yı kullanarak konuları tahmin edin

In [12]:
example = ["My team performed poorly last season. Their best player was out injured and only played one game"]

In [13]:

example_vectorized = vectorizer.transform(example)

topic_distribution = lda.transform(example_vectorized)

print("Topic distribution:", topic_distribution)
print("Predicted topic:", topic_distribution.argmax(axis=1)[0])

Topic distribution: [[0.02027197 0.9194544  0.02007703 0.0201736  0.020023  ]]
Predicted topic: 1


🏁 Tebrikler! LDA'yı hızlı bir şekilde nasıl uygulayacağınızı öğrendiniz.

💾 Not defterinizi `git add/commit/push` yapmayı unutmayın...

🚀 ... ve bir sonraki göreve geçin!